In [ ]:

import ollama
def ask_llm(inp):
    
    response = ollama.chat(
          model = "llama3:8b",
          messages = [{
              "role":"user",
              "content":inp
          }],
        stream = True
    )
    for res in response:
        print(res["message"]["content"],end="",flush=True)
        
    

In [ ]:
while True:
    inp = input("enter your input:")
    ask_llm(inp)

In [ ]:
 pip install langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
FILE_PATH = r"C:\Users\ssram\Downloads\SOP Mohitha.pdf"

loader = PyPDFLoader(FILE_PATH)
docs = loader.load()

print(docs[0].page_content)

In [ ]:
!pip install langchain langchain_community langchainhub chromadb langchain-docling

In [ ]:
!pip install langchain-chroma

In [ ]:
!pip install langchain_huggingface

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 330, chunk_overlap = 20)
texts = text_splitter.split_documents(docs)

In [ ]:
print(texts[0])
print(texts[1])

In [ ]:
!pip install sentence-transformers

In [ ]:
model = "BAAI/bge-large-en-v1.5"

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name = "BAAI/bge-large-en-v1.5")

In [ ]:
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name = "sample_RAG",
    embedding_function=embeddings,
persist_directory = "./chroma_langchain_db",
)

In [ ]:
vector_store._collection.count()

In [ ]:
from langchain_community.vectorstores.utils import filter_complex_metadata
texts = filter_complex_metadata(texts)

In [ ]:
texts[0]

In [ ]:
vector_store.add_documents(texts)

In [ ]:
vector_store._collection.count()

In [ ]:
vector_store._collection.get(ids = ['867371bc-f556-469f-94cc-b378ced3d9e0'], include = ["embeddings","documents"])

In [ ]:
retreiver = vector_store.as_retriever()

In [ ]:
query = "what are my interests?"
retreived_docs = retreiver.invoke(query)
context = "\n\n".join(
    [doc.page_content for doc in retreived_docs]
)
print(context)

prompt = f""" you are a RAG assistant.you need to answer the queries based on the provided context.
if the answer is not present int the provided context then politely return "I dont know".
the input is {query} 
the retreived context is {context} """
ask_llm(prompt)


In [ ]:
prompt

In [ ]:
while True:
    inp = input("Enter input/Query: ")
    retrieved_docs=retreiver.invoke(inp)
    context="\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )
    prompt = f"""
You are a Intelligent RAG assistant and you need to answer quries based on provided context.
If the answer is not present in the provided context then politely return "I donot know"
The input is {inp}
The retrived context is {context}
"""
    ask_llm(prompt)